In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import optuna 
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
train = pd.read_csv('train.csv')


In [2]:
X = train.drop(['Activity'],axis=1)
y = train['Activity']
print(X.head(5))
X_train ,X_valid, y_train, y_valid = train_test_split(X,y,test_size=0.2, random_state= 42, stratify = y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_valid_sc = scaler.transform(X_valid)

         D1        D2    D3   D4        D5        D6        D7        D8  \
0  0.000000  0.497009  0.10  0.0  0.132956  0.678031  0.273166  0.585445   
1  0.366667  0.606291  0.05  0.0  0.111209  0.803455  0.106105  0.411754   
2  0.033300  0.480124  0.00  0.0  0.209791  0.610350  0.356453  0.517720   
3  0.000000  0.538825  0.00  0.5  0.196344  0.724230  0.235606  0.288764   
4  0.100000  0.517794  0.00  0.0  0.494734  0.781422  0.154361  0.303809   

         D9       D10  ...  D1767  D1768  D1769  D1770  D1771  D1772  D1773  \
0  0.743663  0.243144  ...      0      0      0      0      0      0      0   
1  0.836582  0.106480  ...      1      1      1      1      0      1      0   
2  0.679051  0.352308  ...      0      0      0      0      0      0      0   
3  0.805110  0.208989  ...      0      0      0      0      0      0      0   
4  0.812646  0.125177  ...      0      0      0      0      0      0      0   

   D1774  D1775  D1776  
0      0      0      0  
1      0      1   

In [4]:
def calculate_metrics(y_true, y_pred):
    metrics = [accuracy_score, precision_score, recall_score, f1_score]
    names = ["Accuracy", "Precision", "Recall", "F1-Score"]

    results = {}
    for name, metric_fn in zip(names, metrics):
        results[name] = metric_fn(y_true, y_pred)
    return results

In [24]:
model1 = RandomForestClassifier()
model1.fit (X_train_sc,y_train)
prediction1 = model1.predict(X_valid_sc)
scores1 = calculate_metrics (y_valid, prediction1)
for metric, value in scores1.items():
    print(f"{metric}: {value}")


Accuracy: 0.7816245006657789
Precision: 0.7885985748218527
Recall: 0.8157248157248157
F1-Score: 0.8019323671497585


In [27]:
model2 = XGBClassifier(device = 'cuda')
model2.fit(X_train_sc,y_train)
prediction2 = model2.predict(X_valid_sc)
scores2 = calculate_metrics(y_valid,prediction2)
for metric, value in scores2.items():
    print(f"{metric}: {value:}")

Accuracy: 0.7856191744340879
Precision: 0.7873831775700935
Recall: 0.828009828009828
F1-Score: 0.807185628742515


"**Optuna improvement**"

In [40]:
def improve_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "random_state": 42,
        "device": "cuda"
    }

    Model = XGBClassifier(**params)
    Model.fit(X_train_sc,y_train)
    preds = Model.predict(X_valid_sc)
    score = f1_score(y_valid,preds)

    return score
study_xgb = optuna.create_study(directions=['maximize'])
study_xgb.optimize(improve_xgb, n_trials=100)
best_trials = study_xgb.best_trials
for trial in best_trials:
    print(
        f"Trial #{trial.number} | F1: {trial.values} | Params: {trial.params}"
    )

[I 2026-08-01 14:10:35,168] A new study created in memory with name: no-name-bd74dd95-fe8b-467e-9c55-d5acff81443a
[I 2026-08-01 14:10:41,126] Trial 0 finished with value: 0.7857988165680473 and parameters: {'n_estimators': 800, 'learning_rate': 0.002776418642678209, 'max_depth': 6, 'min_child_weight': 5, 'reg_alpha': 3.5857686923946928e-06, 'reg_lambda': 0.032272810154818435, 'gamma': 1.258137562302851e-08}. Best is trial 0 with value: 0.7857988165680473.
[I 2026-08-01 14:10:47,830] Trial 1 finished with value: 0.8092345078979344 and parameters: {'n_estimators': 900, 'learning_rate': 0.07519398312366153, 'max_depth': 9, 'min_child_weight': 1, 'reg_alpha': 1.8059376896477554e-07, 'reg_lambda': 0.1077751367692328, 'gamma': 0.0001399927872004349}. Best is trial 1 with value: 0.8092345078979344.
[I 2026-08-01 14:10:48,283] Trial 2 finished with value: 0.7734282325029656 and parameters: {'n_estimators': 100, 'learning_rate': 0.00970122622112018, 'max_depth': 4, 'min_child_weight': 5, 'reg_a

Trial #92 | F1: [0.8156288156288156] | Params: {'n_estimators': 300, 'learning_rate': 0.3011780779346234, 'max_depth': 4, 'min_child_weight': 2, 'reg_alpha': 1.006967293332709e-06, 'reg_lambda': 3.416185639716747, 'gamma': 1.0754641466697782e-05}


In [41]:


best_model = XGBClassifier(**study_xgb.best_params, random_state=42)
best_model.fit(X_train_sc, y_train)
final_preds = best_model.predict(X_valid_sc)
print(calculate_metrics(y_valid,final_preds))
print(scores2)

{'Accuracy': 0.7922769640479361, 'Precision': 0.8098765432098766, 'Recall': 0.8058968058968059, 'F1-Score': 0.8078817733990148}
{'Accuracy': 0.7856191744340879, 'Precision': 0.7873831775700935, 'Recall': 0.828009828009828, 'F1-Score': 0.807185628742515}


In [42]:
import optuna.visualization as vis
vis.plot_param_importances(study_xgb).show()
vis.plot_optimization_history(study_xgb).show()
vis.plot_parallel_coordinate(study_xgb).show()

In [51]:
def improve_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 5,50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": "log2",
        "random_state": 42,
        "n_jobs": -1,
    }
    Model = RandomForestClassifier(**params)
    Model.fit(X_train_sc,y_train)
    preds = Model.predict(X_valid)
    return f1_score(y_valid, preds)
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(improve_rf, n_trials=100)

[I 2026-08-01 14:22:07,414] A new study created in memory with name: no-name-1d9b5a8c-3d89-4082-9ab3-84d9345c6673
c:\Users\Pizduk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
[I 2026-08-01 14:22:08,061] Trial 0 finished with value: 0.7053726169844021 and parameters: {'n_estimators': 500, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 13}. Best is trial 0 with value: 0.7053726169844021.
c:\Users\Pizduk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
[I 2026-08-01 14:22:08,827] Trial 1 finished with value: 0.715695067264574 and parameters: {'n_estimators': 600, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.71569506726

In [52]:
best_model_rf = RandomForestClassifier(**study_rf.best_params,random_state=42)
best_model_rf.fit(X_train_sc,y_train)
final_preds_rf = best_model_rf.predict(X_valid_sc)
print (calculate_metrics(y_valid,final_preds_rf))
print(scores1)

{'Accuracy': 0.7869507323568575, 'Precision': 0.7933491686460807, 'Recall': 0.8206388206388207, 'F1-Score': 0.8067632850241546}
{'Accuracy': 0.7816245006657789, 'Precision': 0.7885985748218527, 'Recall': 0.8157248157248157, 'F1-Score': 0.8019323671497585}


In [53]:
vis.plot_param_importances(study_rf).show()
vis.plot_optimization_history(study_rf).show()
vis.plot_parallel_coordinate(study_rf).show()

"**Test scores**"


In [58]:
test_df = pd.read_csv("test.csv")
X_test = test_df.copy()
X_test_sc = scaler.transform(X_test)
test_preds = best_model.predict(X_test_sc)
test_probs = best_model.predict_proba(X_test_sc)[:1]
print(test_probs)

[[0.01471084 0.98528916]]
